# 08 · Lazy composition with Dask

Large cubes should not be loaded merely because a workflow was described. This
notebook constructs a chunked xarray cube, verifies that verbs build a lazy
graph, and computes only the small final map needed for the plot.

In [ ]:
import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

# Start with ordinary NumPy values, then wrap them in a Dask array split into
# bounded chunks: 6 time steps × 4 rows × 5 columns per chunk.
rng = np.random.default_rng(7)
values = rng.normal(size=(24, 8, 10)).astype("float32")
lazy_values = da.from_array(values, chunks=(6, 4, 5))

# xarray adds scientific coordinates without changing the lazy Dask backing.
cube = xr.DataArray(
    lazy_values,
    dims=("time", "y", "x"),
    coords={
        "time": pd.date_range("2023-01-01", periods=24, freq="MS"),
        "y": np.arange(8),
        "x": np.arange(10),
    },
    name="signal",
    attrs={"source": "deterministic synthetic vignette"},
)

# These verbs construct a task graph; neither one requests concrete values.
# The final result is a 2D variance map because the time dimension is reduced.
result = (
    pipe(cube)
    | v.anomaly(dim="time")
    | v.variance(dim="time", keep_dim=False)
).unwrap()

# chunks proves the arrays remain lazy. Counting graph tasks is safe because it
# inspects the plan rather than computing the array values.
assert cube.chunks is not None
assert result.chunks is not None
graph_tasks = len(result.data.__dask_graph__())

# compute() is the intentional execution boundary. Only the small final map is
# materialized, and that map—not the full source cube—is sent to Matplotlib.
materialized = result.compute()
assert materialized.chunks is None
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
materialized.plot(ax=ax, cmap="magma", cbar_kwargs={"label": "anomaly variance"})
ax.set_title(f"Computed final map · graph previously had {graph_tasks} tasks")
plt.show()